In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain.agents import create_agent

llm = ChatAnthropic(
     api_key=os.getenv("ANTHROPIC_API_KEY"),
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_ENDPOINT")
)
print("Setup complete!")

/home/ubuntu/Desktop/GenAi/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete!


In [2]:
import math
from datetime import datetime

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Supports basic operations (+, -, *, /, **) and functions like sqrt, sin, cos.
    Examples: '2 + 2', '10 ** 2', 'sqrt(16)', '100 / 4'"""
    try:
        # Safe evaluation with limited functions
        allowed_names = {
            'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos,
            'tan': math.tan, 'log': math.log, 'pi': math.pi, 'e': math.e
        }
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_current_datetime() -> str:
    """Get the current date and time. Use this when the user asks about today's date or current time."""
    now = datetime.now()
    return f"Current date and time: {now.strftime('%A, %B %d, %Y at %I:%M %p')}"

@tool
def search_knowledge_base(query: str) -> str:
    """Search our internal knowledge base for information about AI, Machine Learning, and LangChain."""
    # Simulated knowledge base
    knowledge = {
        "langchain": "LangChain is an open-source framework for building LLM applications. Created by Harrison Chase in 2022. Key components: Chains, Agents, Memory, Tools.",
        "agentic ai": "Agentic AI refers to AI systems that can plan, reason, and take autonomous actions to achieve goals. Key features: tool use, memory, multi-step reasoning.",
        "claude": "Claude is an AI assistant created by Anthropic. Known for being helpful, harmless, and honest. Supports up to 200K context window.",
        "react": "ReAct (Reason + Act) is a prompting pattern where the AI thinks step-by-step, takes actions, observes results, and repeats until done.",
        "rag": "RAG (Retrieval-Augmented Generation) combines LLMs with external knowledge retrieval. Steps: Query → Retrieve documents → Generate answer with context."
    }
    
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return f"Knowledge Base Result:\n{value}"
    return "No matching information found in the knowledge base."

# Collect all tools
tools = [calculator, get_current_datetime, search_knowledge_base]
print("Tools defined:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

Tools defined:
  - calculator: Evaluate a mathematical expression. Supports basic operation...
  - get_current_datetime: Get the current date and time. Use this when the user asks a...
  - search_knowledge_base: Search our internal knowledge base for information about AI,...


In [3]:
system_prompt = """You are a helpful research assistant with access to tools.

Your capabilities:
- Calculate mathematical expressions
- Get the current date and time
- Search our knowledge base about AI topics

Guidelines:
1. Always use tools when you need factual information
2. Think step-by-step before answering complex questions
3. Cite your sources when using the knowledge base
4. If you don't know something, say so honestly
"""

print("System prompt created!")

System prompt created!


In [4]:
# Create the agent using LangChain 1.x API
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("Agent created!")

Agent created!


In [5]:
chat_history = []  # Our memory store

def chat(user_input: str) -> str:
    """Send a message to the agent and update memory."""
    global chat_history
    
    # Build messages list: chat history + current input
    messages = chat_history + [HumanMessage(content=user_input)]
    
    # Invoke the agent with messages
    result = agent.invoke({"messages": messages})
    
    # Extract the response (it's the last message in the result)
    response_text = result["messages"][-1].content
    
    # Update memory
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response_text))
    
    return response_text

print("Chat function ready!")

Chat function ready!


In [6]:
# Test 1: Calculator tool
print("="*60)
print("Test 1: Math calculation")
print("="*60)
response = chat("What is 15% of 250?")
print(f"\nFinal Answer: {response}")

Test 1: Math calculation

Final Answer: 15% of 250 is **37.5**.


In [7]:
# Test 2: Knowledge base search
print("="*60)
print("Test 2: Knowledge lookup")
print("="*60)
response = chat("What is LangChain and who created it?")
print(f"\nFinal Answer: {response}")

Test 2: Knowledge lookup

Final Answer: **LangChain** is an open-source framework designed for building applications powered by Large Language Models (LLMs). 

**Creator:** LangChain was created by **Harrison Chase** in 2022.

**Key Components:** The framework includes several important components:
- **Chains**: Sequences of calls to language models and other tools
- **Agents**: Systems that can use tools to accomplish tasks
- **Memory**: Functionality to maintain conversation context and state
- **Tools**: Integrations with external systems and APIs

LangChain has become a popular framework in the AI/ML community for streamlining the development of LLM-based applications and making it easier to work with language models in production environments.


In [8]:
# Test 3: Multi-tool usage
print("="*60)
print("Test 3: Complex question requiring multiple tools")
print("="*60)
response = chat("What is today's date, and if LangChain was created in 2022, how many years ago was that?")
print(f"\nFinal Answer: {response}")

Test 3: Complex question requiring multiple tools

Final Answer: **Today's date:** Sunday, June 28, 2026 at 10:53 PM

**Years since LangChain was created:** LangChain was created in 2022, which was **4 years ago** (2026 - 2022 = 4 years).


In [9]:
# Test 4: Memory test
print("="*60)
print("Test 4: Memory - follow-up question")
print("="*60)
response = chat("Based on what you told me earlier, what are the key components of LangChain?")
print(f"\nFinal Answer: {response}")

Test 4: Memory - follow-up question

Final Answer: Based on what I told you earlier, the key components of LangChain are:

1. **Chains** - Sequences of calls to language models and other tools

2. **Agents** - Systems that can use tools to accomplish tasks

3. **Memory** - Functionality to maintain conversation context and state

4. **Tools** - Integrations with external systems and APIs

These components work together to help developers build sophisticated applications powered by Large Language Models.


In [10]:
from langchain_core.tools import tool
 
@tool
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """
    Convert between units.
    Supports:
    - Temperature: celsius, fahrenheit, kelvin
    - Length: meters, feet, kilometers, miles
    - Weight: kg, pounds
    """
 
    from_unit = from_unit.lower()
    to_unit = to_unit.lower()
 
    # Temperature
    if from_unit == "celsius" and to_unit == "fahrenheit":
        result = (value * 9/5) + 32
    elif from_unit == "fahrenheit" and to_unit == "celsius":
        result = (value - 32) * 5/9
    elif from_unit == "celsius" and to_unit == "kelvin":
        result = value + 273.15
    elif from_unit == "kelvin" and to_unit == "celsius":
        result = value - 273.15
 
    # Length
    elif from_unit == "meters" and to_unit == "feet":
        result = value * 3.28084
    elif from_unit == "feet" and to_unit == "meters":
        result = value / 3.28084
    elif from_unit == "kilometers" and to_unit == "miles":
        result = value * 0.621371
    elif from_unit == "miles" and to_unit == "kilometers":
        result = value / 0.621371
 
    # Weight
    elif from_unit == "kg" and to_unit == "pounds":
        result = value * 2.20462
    elif from_unit == "pounds" and to_unit == "kg":
        result = value / 2.20462
 
    else:
        return "Unsupported conversion."
 
    return f"{value} {from_unit} = {round(result, 2)} {to_unit}"

In [11]:
result = convert_units.invoke({
    "value": 100,
    "from_unit": "celsius",
    "to_unit": "fahrenheit"
})
print(result)

100.0 celsius = 212.0 fahrenheit


In [12]:
system_prompt = """
You are a professional Financial Advisor AI assistant.
 
Your responsibilities:
- Provide financial guidance in a clear, professional, and formal manner.
- Help users with budgeting, savings, investments, taxes, loans, interest calculations, and financial planning.
- Use available tools whenever calculations are required.
- Explain calculations step by step.
- If information is uncertain, clearly mention the limitations.
 
Guidelines:
1. Always use formal and respectful language.
2. Always include a disclaimer that your advice is for educational purposes and not professional financial advice.
3. Encourage users to consult a certified financial advisor before making major financial decisions.
4. Provide accurate calculations and easy-to-understand explanations.
5. Never make unrealistic financial promises.
 
Disclaimer:
This information is provided for educational purposes only and should not be considered professional financial advice. Please consult a certified financial advisor before making investment or financial decisions.
"""
 
print("Financial advisor system prompt created!")

Financial advisor system prompt created!


In [13]:
financial_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)
 
print("Financial advisor agent created!")

Financial advisor agent created!


In [14]:
response = financial_agent.invoke({
    "messages": [
        HumanMessage(content="If I invest ₹50,000 at 8% annual interest for 3 years, what will be the approximate amount?")
    ]
})
 
print(response["messages"][-1].content)

## Investment Calculation Summary

**Principal Amount:** ₹50,000  
**Annual Interest Rate:** 8%  
**Investment Period:** 3 years  
**Calculation Method:** Compound Interest

### Result:
**Approximate Final Amount: ₹62,985.60**

### Breakdown:

The compound interest formula used is:
**A = P(1 + r)^n**

Where:
- A = Final Amount
- P = Principal (₹50,000)
- r = Annual interest rate (0.08 or 8%)
- n = Number of years (3)

**Step-by-step calculation:**
- Year 1: ₹50,000 × 1.08 = ₹54,000
- Year 2: ₹54,000 × 1.08 = ₹58,320
- Year 3: ₹58,320 × 1.08 = ₹62,985.60

**Your Profit/Gain:** ₹62,985.60 - ₹50,000 = **₹12,985.60**

### Important Notes:
- This calculation assumes the interest is compounded annually.
- Different investments (Fixed Deposits, Bonds, Mutual Funds, etc.) may have different compounding periods (quarterly, monthly, daily).
- This calculation does not account for taxes, which may be applicable depending on your investment type and local regulations.
- Actual returns may vary bas